# MeRIP-seq Peak Calling Method

In [1]:
data_dir <- "~/dataset_1/bulk_omics/meripseq_Hs_HCCLM3_ALKBH5_GSE149510/02_output"

gtf_file <- "~/reference/Homo_sapiens/hg38/gencode.v48.annotation.gtf"

In [2]:
# # load reference gene annotation
# txdb <- txdbmaker::makeTxDbFromGFF(
#     gtf_file, 
#     dataSource = "GENCODE", 
#     organism = "Homo sapiens", 
#     taxonomyId = 9606, 
#     metadata = data.frame(
#         name = c("Genome", "GENCODE Release"), 
#         value = c("hg38", "v48")))

# gtf <- rtracklayer::import(gtf_file) |>
#     GenomeInfoDb::keepStandardChromosomes(pruning.mode = "coarse")

# txdb <- txdbmaker::makeTxDbFromGRanges(
#     gtf, 
#     taxonomyId = 9606, 
#     drop.stop.codons = TRUE, 
#     metadata = data.frame(
#         name = c("Genome", "GENCODE Release"), 
#         value = c("hg38", "v48")))

# 1. [exomePeak2](https://github.com/ZW-xjtlu/exomePeak2)

> 1. Check and index the BAM files with scanMeripBAM.
> 2. Call modification peaks on exons with **exomePeakCalling**.
> 3. Calculate offset factors of GC content biases with normalizeGC.
> 4. Calculate (differential) modification statistics with the generalized linear model (GLM) using glmM or glmDM.
> 5. Export the peaks/sites statistics with user defined format by exportResults.
> - Jingxian Zhou, Zhen Wei, Di Zhen, Yue Wang, Jionglong Su, Frans Coenen, Guifang Jia, Daniel J Rigden, Jia Meng, Comprehensive Epitranscriptome Analysis from MeRIP-seq Data with exomePeak2, Genomics, Proteomics & Bioinformatics, 2026;, qzag019, https://doi.org/10.1093/gpbjnl/qzag019.

In [3]:
# if (!requireNamespace("BiocManager", quietly = TRUE))
#     install.packages("BiocManager")

# BiocManager::install(c("Rsamtools", "GenomicAlignments", "GenomicRanges", 
#                        "GenomicFeatures", "DESeq2", "ggplot2", "mclust", "BSgenome", 
#                        "Biostrings", "GenomeInfoDb", "BiocParallel", "IRanges", 
#                        "S4Vectors", "rtracklayer", "methods", "stats", 
#                        "utils", "BiocGenerics", "magrittr", "speedglm", "splines"))

# if (!requireNamespace("remotes", quietly = TRUE))
#     install.packages("remotes")

# remotes::install_github("ZW-xjtlu/exomePeak2", build_vignettes = TRUE)

In [4]:
suppressPackageStartupMessages(library(exomePeak2))

In [5]:
txdb <- txdbmaker::makeTxDbFromGFF(gtf_file)

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
“The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored.”
Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
“genome version information is not available for this TxDb object”
OK



In [9]:
IP_BAM <- file.path(data_dir, "04_bam", "HCCLM3_ALKBH5_NC_rep1_m6A.bam")
INPUT_BAM <- file.path(data_dir, "04_bam", "HCCLM3_ALKBH5_NC_rep1_Input.bam")

In [10]:
output_dir <- "~/dataset_1/bulk_omics/meripseq_Hs_HCCLM3_ALKBH5_GSE149510"
prefix <- "ALKBH5_NC_rep1"

In [ ]:
# ~ 20 min
peaks <- exomePeak2(
    bam_ip = IP_BAM, 
    bam_input = INPUT_BAM, 
    txdb = txdb, 
    genome = "hg38", 
    strandness = "unstrand", 
    fragment_length = 100, 
    bin_size = 25, 
    step_size = 25, 
    p_cutoff = 1e-5, 
    parallel = 1, 
    save_dir = output_dir, 
    experiment_name = prefix
)

Extract bin features ... 
OK

Count reads on bin features ... 
OK

Identify background features ... 
OK

Estimate sample sepecific size factors from the background ... 
OK

Calculate bin GC contents on exons ... 
OK

Fit GC curves with smoothing splines ... 
OK

Calculate offset matrix ... 
OK

Detect peaks with GLM ... 
Warning message in qf(0.99, p, m - p):
“NaNs produced”
OK



In [14]:
str(peaks)

Formal class 'CompressedGRangesList' [package "GenomicRanges"] with 5 slots
  ..@ unlistData     :Formal class 'GRanges' [package "GenomicRanges"] with 7 slots
  .. .. ..@ seqnames       :Formal class 'Rle' [package "S4Vectors"] with 4 slots
  .. .. .. .. ..@ values         : Factor w/ 25 levels "chr1","chr2",..: 20 1 6 1 23 7 12 11 4 7 ...
  .. .. .. .. ..@ lengths        : int [1:6432] 15 27 38 8 10 15 5 7 10 8 ...
  .. .. .. .. ..@ elementMetadata: NULL
  .. .. .. .. ..@ metadata       : list()
  .. .. ..@ ranges         :Formal class 'IRanges' [package "IRanges"] with 6 slots
  .. .. .. .. ..@ start          : int [1:62764] 50958965 50958865 50957146 50958363 50955171 50945824 50948629 50943261 50945737 50941190 ...
  .. .. .. .. ..@ width          : int [1:62764] 175 75 8 167 125 100 50 13 37 20 ...
  .. .. .. .. ..@ NAMES          : chr [1:62764] "1" "2" "3" "3" ...
  .. .. .. .. ..@ elementType    : chr "ANY"
  .. .. .. .. ..@ elementMetadata: NULL
  .. .. .. .. ..@ metadata    

> Annotations on the columns of the output table:
> - chr: the chromosome name of the peak.
> - chromStart: the start of the peak on the chromosome.
> - chromEnd: the end of the peak on the chromosome.
> - name: the unique ID of the modification peak.
> - strand: the strand of the peak on genome.
> - blockCount: the block (exon) number within the peak.
> - blockSizes: the widths of the blocks.
> - blockStarts: the start positions of the blocks.
> - geneID: the gene ID of the peak; multiple gene IDs will be returned when peak is compatible with > 1 genes.
> - RPM.input: the peak RPM (Reads per million mapped reads) calculated from the pooled input samples.
> - RPM.IP: the peak RPM calculated from the pooled IP samples.
> - log2FC: the estimate of IP over input log2 fold change (coefficient estimates of βi,1).
> - pvalue: the p-value of the peak, which is calculated by the Wald test over the coefficient of βi,1.
> - fdr: the adjusted Wald test p-value using Benjamini Hochberg approach.
> - score: the -log10 p value of the peak.

# 2. [exomePeak](https://github.com/ZW-xjtlu/exomePeak)

> - Meng J, Lu Z, Liu H, Zhang L, Zhang S, Chen Y, Rao MK, Huang Y. A protocol for RNA methylation differential analysis with MeRIP-Seq data and exomePeak R/Bioconductor package. Methods. 2014 Oct 1;69(3):274-81. doi: 10.1016/j.ymeth.2014.06.008. Epub 2014 Jun 27.

In [15]:
# if (!requireNamespace("BiocManager", quietly = TRUE))
#     install.packages("BiocManager")

# BiocManager::install(c("SummarizedExperiment","cqn","Rsamtools",
#                        "GenomicAlignments","GenomicRanges","GenomicFeatures",
#                        "DESeq2","ggplot2","mclust",
#                        "genefilter","BSgenome","BiocParallel",
#                        "IRanges","S4Vectors","quantreg",
#                        "reshape2","rtracklayer","apeglm","RMariaDB"))

# if (!requireNamespace("remotes", quietly = TRUE))
#     install.packages("remotes")

# remotes::install_github("ZW-xjtlu/exomePeak")

In [16]:
suppressPackageStartupMessages(library(exomePeak))

In [17]:
txdb <- txdbmaker::makeTxDbFromGFF(gtf_file)

Import genomic features from the file as a GRanges object ... 
OK

Prepare the 'metadata' data frame ... 
OK

Make the TxDb object ... 
Warning message in .get_cds_IDX(mcols0$type, mcols0$phase):
“The "phase" metadata column contains non-NA values for features of type
  stop_codon. This information was ignored.”
Warning message in .makeTxDb_normarg_chrominfo(chrominfo):
“genome version information is not available for this TxDb object”
OK



In [18]:
IP_BAM <- file.path(data_dir, "04_bam", "HCCLM3_ALKBH5_NC_rep1_m6A.bam")
INPUT_BAM <- file.path(data_dir, "04_bam", "HCCLM3_ALKBH5_NC_rep1_Input.bam")

In [19]:
output_dir <- "~/dataset_1/bulk_omics/meripseq_Hs_HCCLM3_ALKBH5_GSE149510"
prefix <- "ALKBH5_NC_rep1"

In [ ]:
# ~ 180 min
peaks <- exomepeak(
    IP_BAM = IP_BAM, 
    INPUT_BAM = INPUT_BAM, 
    TXDB = txdb, 
    OUTPUT_DIR = output_dir, 
    EXPERIMENT_NAME = prefix, 
    PEAK_CUTOFF_PVALUE = 1e-5, 
    PEAK_CUTOFF_FDR = 0.01, 
    FOLD_ENRICHMENT = 1, 
    FRAGMENT_LENGTH = 100, 
    READ_LENGTH = NA, 
    MINIMAL_MAPQ = 30
)

'select()' returned 1:many mapping between keys and columns



[1] "Divide transcriptome into chr-gene-batch sections ..."
[1] "Get Reads Count ..."
[1] "This step may take a few hours ..."
[1] "0.212 %"
[1] "0.425 %"
[1] "0.637 %"
[1] "0.849 %"
[1] "1.06 %"
[1] "1.27 %"
[1] "1.49 %"
[1] "1.7 %"
[1] "1.91 %"
[1] "2.12 %"
[1] "2.34 %"
[1] "2.55 %"
[1] "2.76 %"
[1] "2.97 %"
[1] "3.18 %"
[1] "3.4 %"
[1] "3.61 %"
[1] "3.82 %"
[1] "4.03 %"
[1] "4.25 %"
[1] "4.46 %"
[1] "4.67 %"
[1] "4.88 %"
[1] "5.1 %"
[1] "5.31 %"
[1] "5.52 %"
[1] "5.73 %"
[1] "5.94 %"
[1] "6.16 %"
[1] "6.37 %"
[1] "6.58 %"
[1] "6.79 %"
[1] "7.01 %"
[1] "7.22 %"
[1] "7.43 %"
[1] "7.64 %"
[1] "7.86 %"
[1] "8.07 %"
[1] "8.28 %"
[1] "8.49 %"
[1] "8.7 %"
[1] "8.92 %"
[1] "9.13 %"
[1] "9.34 %"
[1] "9.55 %"
[1] "9.77 %"
[1] "9.98 %"
[1] "10.2 %"
[1] "10.4 %"
[1] "10.6 %"
[1] "10.8 %"
[1] "11 %"
[1] "11.3 %"
[1] "11.5 %"
[1] "11.7 %"
[1] "11.9 %"
[1] "12.1 %"
[1] "12.3 %"
[1] "12.5 %"
[1] "12.7 %"
[1] "13 %"
[1] "13.2 %"
[1] "13.4 %"
[1] "13.6 %"
[1] "13.8 %"
[1] "14 %"
[1] "14.2 %"
[1] "14.

ERROR: Error in call_fun_in_txdbmaker("makeTxDb", ...): makeTxDb() has moved from GenomicFeatures to the txdbmaker package, and is
  formally defunct in GenomicFeatures >= 1.61.1. Please call
  txdbmaker::makeTxDb() to get rid of this error.


# 3. [MeTPeak](https://github.com/compgenomics/MeTPeak)

> - Xiaodong Cui, Jia Meng, Shaowu Zhang, Yidong Chen, Yufei Huang, A novel algorithm for calling mRNA m 6 A peaks by modeling biological variances in MeRIP-seq data , Bioinformatics, Volume 32, Issue 12, June 2016, Pages i378–i385, https://doi.org/10.1093/bioinformatics/btw281.

In [ ]:
# devtools::install_github(
#     "compgenomics/MeTPeak", build_opts = c("--no-resave-data", "--no-manual"))

In [ ]:
# suppressPackageStartupMessages(library(MeTPeak))

In [ ]:
# peaks <- metpeak(
#     IP_BAM = IP_BAM, 
#     INPUT_BAM = INPUT_BAM, 
#     GENE_ANNO_GTF = gtf_dir, 
#     EXPERIMENT_NAME = prefix, 
#     FRAGMENT_LENGTH = 100, 
#     FOLD_ENRICHMENT = 1,
#     PEAK_CUTOFF_PVALUE = 1e-5, 
#     PEAK_CUTOFF_FDR = 0.05
# )

# 4. [TRESS](https://github.com/ZhenxingGuo0015/TRESS)

In [ ]:
# devtools::install_github(
#     "https://github.com/ZhenxingGuo0015/TRESS", build_vignettes = TRUE)

In [ ]:
# suppressPackageStartupMessages(library(TRESS))

In [ ]:
# txdb_sqlite <- file.path("~/reference/Homo_sapiens/hg38", "gencode.v48.sqlite")
# AnnotationDbi::saveDb(txdb, file = txdb_sqlite)